# Final analysis: predictive quality and energy trade-offs

This notebook reads only saved artifacts. It isolates measured pairs with an absolute V0/V1 energy difference greater than 50 J, summarizes grouped out-of-fold predictions, compares model metrics with measured training/inference energy, and calculates break-even counts only when actual sensor measurements are available. Estimates and unavailable measurements are not presented as measurements.


In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebooks 01–05 first to create the processed analysis files.")
PROCESSED_DIR = ROOT / "Data" / "Processed"
RESULTS_ROOT = ROOT / "Results"
RESULTS_DIR = RESULTS_ROOT / "validated"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

validated = pd.read_parquet(PROCESSED_DIR / "validated_energy_results.parquet")
performance = pd.read_csv(RESULTS_DIR / "model_performance.csv")
oof = pd.read_parquet(RESULTS_DIR / "model_oof_predictions.parquet")
measurements = pd.read_csv(RESULTS_DIR / "energy_measurements.csv")
measurement_status = json.loads((RESULTS_DIR / "energy_measurement_status.json").read_text(encoding="utf-8"))

validated["energy_diff_v0_minus_v1"] = validated["result_energy_v0"] - validated["result_energy_v1"]
validated["absolute_energy_difference_j"] = validated["energy_diff_v0_minus_v1"].abs()
large_difference_pairs = validated.loc[validated["absolute_energy_difference_j"] > 50].copy()
large_difference_pairs["lower_energy_version"] = np.where(
    large_difference_pairs["energy_diff_v0_minus_v1"] > 0, "V1", "V0"
)
pair_columns = [
    "problem_id", "submission_id_v0", "submission_id_v1", "language",
    "result_energy_v0", "result_energy_v1", "energy_diff_v0_minus_v1",
    "absolute_energy_difference_j", "lower_energy_version",
]
large_difference_pairs[pair_columns].to_csv(RESULTS_DIR / "large_energy_difference_pairs.csv", index=False)
large_difference_summary = pd.DataFrame([{
    "validated_pairs": len(validated),
    "pairs_abs_difference_over_50_j": len(large_difference_pairs),
    "share_over_50_j": len(large_difference_pairs) / len(validated) if len(validated) else np.nan,
    "mean_absolute_difference_over_50_j": large_difference_pairs["absolute_energy_difference_j"].mean(),
    "median_absolute_difference_over_50_j": large_difference_pairs["absolute_energy_difference_j"].median(),
    "v1_lower_energy_count_over_50_j": int((large_difference_pairs["lower_energy_version"] == "V1").sum()),
    "v0_lower_energy_count_over_50_j": int((large_difference_pairs["lower_energy_version"] == "V0").sum()),
}])
large_difference_summary.to_csv(RESULTS_DIR / "large_energy_difference_summary.csv", index=False)
display(large_difference_summary)


## Model quality, recommendation energy impact, and break-even

Energy savings below compare each model's held-out recommendation with always choosing V0. Break-even is reported only when Notebook 05 contains actual meter-backed measurements.


In [ ]:
model_energy_rows = []
break_even_rows = []
actual_measurements = (
    not measurements.empty
    and measurements["actual_measurement"].fillna(False).astype(bool).all()
    and bool(measurement_status.get("actual_measurements_available"))
)
measurement_metadata = json.loads(
    (PROCESSED_DIR / "models" / "validated" / "training_metadata.json").read_text(encoding="utf-8")
)
measurement_feature_config = measurement_metadata["feature_config"]
for (model_name, feature_config), predictions in oof.groupby(["model", "feature_config"]):
    predictions = predictions.copy()
    predictions["absolute_energy_difference_j"] = (
        predictions["result_energy_v0"] - predictions["result_energy_v1"]
    ).abs()
    large_oof = predictions[predictions["absolute_energy_difference_j"] > 50]
    chosen_energy = np.where(
        large_oof["y_pred"].to_numpy() == 1,
        large_oof["result_energy_v1"].to_numpy(),
        large_oof["result_energy_v0"].to_numpy(),
    )
    fixed_v0_energy = np.where(
        large_oof["was_swapped"].to_numpy(dtype=bool),
        large_oof["result_energy_v1"].to_numpy(),
        large_oof["result_energy_v0"].to_numpy(),
    )
    saving_vs_v0 = fixed_v0_energy - chosen_energy
    model_energy_rows.append({
        "model": model_name,
        "feature_config": feature_config,
        "oof_pairs_over_50_j": len(large_oof),
        "mean_recommended_energy_j": float(np.mean(chosen_energy)) if len(chosen_energy) else np.nan,
        "mean_saving_vs_always_v0_j": float(np.mean(saving_vs_v0)) if len(saving_vs_v0) else np.nan,
        "total_saving_vs_always_v0_j": float(np.sum(saving_vs_v0)) if len(saving_vs_v0) else np.nan,
    })

    task_energy = (
        measurements[measurements["model"] == model_name]
        if actual_measurements and feature_config == measurement_feature_config
        else pd.DataFrame()
    )
    training = task_energy[task_energy["task"] == "training"] if not task_energy.empty else pd.DataFrame()
    inference = task_energy[task_energy["task"] == "inference"] if not task_energy.empty else pd.DataFrame()
    if actual_measurements and not training.empty and not inference.empty and len(large_oof):
        training_energy_j = float(training["net_energy_joules"].mean())
        inference_batch_energy_j = float(inference["net_energy_joules"].mean())
        n_inference_pairs = float(inference["n_inference_pairs"].mean())
        inference_per_pair_j = inference_batch_energy_j / n_inference_pairs
        gross_saving_per_pair_j = float(np.mean(saving_vs_v0))
        net_saving_per_pair_j = gross_saving_per_pair_j - inference_per_pair_j
        break_even = math.ceil(training_energy_j / net_saving_per_pair_j) if net_saving_per_pair_j > 0 else None
        reason = "finite" if break_even is not None else "No positive net saving per recommended pair."
    else:
        training_energy_j = inference_batch_energy_j = inference_per_pair_j = np.nan
        gross_saving_per_pair_j = float(np.mean(saving_vs_v0)) if len(saving_vs_v0) else np.nan
        net_saving_per_pair_j = np.nan
        break_even = None
        reason = "Unavailable: no actual hardware energy measurements for both training and inference."
    break_even_rows.append({
        "model": model_name,
        "feature_config": feature_config,
        "training_energy_j_per_fit": training_energy_j,
        "inference_energy_j_per_measured_batch": inference_batch_energy_j,
        "inference_pairs_per_batch": float(inference["n_inference_pairs"].mean()) if actual_measurements and not inference.empty else np.nan,
        "inference_energy_j_per_pair": inference_per_pair_j,
        "mean_saving_vs_always_v0_j_per_pair_over_50_j": gross_saving_per_pair_j,
        "net_saving_j_per_pair_after_inference": net_saving_per_pair_j,
        "break_even_recommendations": break_even,
        "status": reason,
    })

model_energy = pd.DataFrame(model_energy_rows)
break_even_analysis = pd.DataFrame(break_even_rows)
model_energy.to_csv(RESULTS_DIR / "model_energy_impacts.csv", index=False)
break_even_analysis.to_csv(RESULTS_DIR / "break_even_analysis.csv", index=False)
display(performance.merge(model_energy, on=["model", "feature_config"], how="left"))
display(break_even_analysis)
print("Measurement status:", json.dumps(measurement_status, indent=2))


## Final figures

Validated-data figures are saved in Results/validated/. The energy-versus-performance chart uses sensor-backed energy rows only.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(large_difference_pairs["energy_diff_v0_minus_v1"], bins=50, ax=ax, color="darkcyan")
ax.axvline(0, color="black", linestyle="--")
ax.set(
    title="V0 − V1 energy differences for pairs over 50 J",
    xlabel="Energy difference (Joules)",
    ylabel="Pairs",
)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "final_large_difference_energy.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=performance, x="model", y="balanced_accuracy_mean", ax=axes[0], color="slateblue")
axes[0].set(title="GroupKFold balanced accuracy", xlabel="Model", ylabel="Mean balanced accuracy")
axes[0].tick_params(axis="x", rotation=25)
if actual_measurements:
    measured_energy = measurements.pivot_table(
        index="model", columns="task", values="net_energy_joules", aggfunc="mean"
    ).reset_index()
    measured_energy["fit_plus_inference_batch_joules"] = (
        measured_energy["training"] + measured_energy["inference"]
    )
    performance_energy = performance.merge(measured_energy, on="model", how="inner")
    sns.scatterplot(
        data=performance_energy,
        x="balanced_accuracy_mean",
        y="fit_plus_inference_batch_joules",
        hue="model",
        s=90,
        ax=axes[1],
    )
    axes[1].set(
        title=f"Performance vs. measured compute energy ({measurement_status.get('backend')})",
        xlabel="GroupKFold balanced accuracy",
        ylabel="Training fit + inference batch energy (Joules)",
    )
else:
    axes[1].text(0.5, 0.5, "No actual hardware energy measurements available", ha="center", va="center")
    axes[1].set_axis_off()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "model_performance_vs_energy.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved final tables and figures to {RESULTS_DIR}.")


## Compare validated and significant data

This combines the saved cross-validation results and energy summaries from the two dataset folders. The comparison is descriptive because the datasets have different sizes and may have different distributions.


In [ ]:
comparison_dir = RESULTS_ROOT / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
performance_frames, energy_frames = [], []
for dataset_name in ("validated", "significant"):
    dataset_dir = RESULTS_ROOT / dataset_name
    for filename in ("model_performance.csv", "codebert_performance.csv"):
        path = dataset_dir / filename
        if path.is_file():
            frame = pd.read_csv(path)
            frame.insert(0, "dataset", dataset_name)
            performance_frames.append(frame)
    for filename in ("energy_measurement_summary.csv", "codebert_energy_measurement_summary.csv"):
        path = dataset_dir / filename
        if path.is_file():
            frame = pd.read_csv(path)
            frame.insert(0, "dataset", dataset_name)
            energy_frames.append(frame)
if performance_frames:
    performance_comparison = pd.concat(performance_frames, ignore_index=True)
    performance_comparison.to_csv(comparison_dir / "model_performance_comparison.csv", index=False)
    display(performance_comparison)
if energy_frames:
    energy_comparison = pd.concat(energy_frames, ignore_index=True)
    energy_comparison.to_csv(comparison_dir / "energy_measurement_comparison.csv", index=False)
    display(energy_comparison)
print(f"Saved cross-dataset comparisons under {comparison_dir}.")
